# Day 4 v2 — Model 01: DNN + TF-IDF char_wb 100K

**Architecture:** TF-IDF char_wb (2,4) 100K features → sparse mini-batch → PriceDNN (8 ResidualBlocks, hidden=4096)

**Target:** MAE < 80k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

**Key technique:** SparseDataset converts rows on demand — no `.toarray()` on full 269K × 100K matrix.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.deep_neural_network_sparse import SparseDNNRunner

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 3090 Ti


## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 269,112 | Val: 3,926 | Test: 3,872


## 2. Setup Model

Fit TF-IDF char_wb (2,4) 100K trên 269K docs. Config nhất quán với Day 3 v2 model 5A (MAE=92.6k).

In [3]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)

runner = SparseDNNRunner(train, val)
runner.setup(vectorizer, batch_size=64, num_blocks=8, hidden_size=4096)

Fitting vectorizer on 269K train docs...
Feature matrix: (269112, 100000)
PriceDNN: 678,248,449 trainable params | input_size=100000
Using cuda


## 3. Train

Max 10 epochs, early stopping patience=3. Val feedback dùng val[:1000] mỗi epoch.

In [4]:
history = runner.train(epochs=10, patience=3)

Epoch 1/10 | train_loss=0.5898 | val_loss=0.4127 | val_mae=102.83k | lr=0.000976
  ** best val_mae=102.83k


Epoch 2/10 | train_loss=0.2364 | val_loss=0.3546 | val_mae=88.97k | lr=0.000905
  ** best val_mae=88.97k


Epoch 3/10 | train_loss=0.1560 | val_loss=0.3627 | val_mae=89.75k | lr=0.000794


Epoch 4/10 | train_loss=0.1154 | val_loss=0.3541 | val_mae=88.01k | lr=0.000655
  ** best val_mae=88.01k


Epoch 5/10 | train_loss=0.0881 | val_loss=0.3420 | val_mae=85.10k | lr=0.000500
  ** best val_mae=85.10k


Epoch 6/10 | train_loss=0.0677 | val_loss=0.3434 | val_mae=85.46k | lr=0.000345


Epoch 7/10 | train_loss=0.0523 | val_loss=0.3458 | val_mae=85.01k | lr=0.000206
  ** best val_mae=85.01k


Epoch 8/10 | train_loss=0.0408 | val_loss=0.3408 | val_mae=85.22k | lr=0.000095


Epoch 9/10 | train_loss=0.0324 | val_loss=0.3409 | val_mae=84.77k | lr=0.000024
  ** best val_mae=84.77k


Epoch 10/10 | train_loss=0.0272 | val_loss=0.3405 | val_mae=84.57k | lr=0.000000
  ** best val_mae=84.57k


## 4. Training History

In [5]:
plot_training_history(history, title="DNN + TF-IDF char_wb 100K")

## 5. Save Weights + Val Predictions + Test Predictions

In [6]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/dnn_tfidf.pth")
print("Saved weights/dnn_tfidf.pth")

Path("val_predictions").mkdir(exist_ok=True)

# Val predictions — full 3926 samples for stacking (Task 7)
print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/dnn_tfidf_val.json", "w") as f:
    json.dump(val_preds, f)

# Test predictions — full 3872 samples for stacking
print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/dnn_tfidf_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

Saved weights/dnn_tfidf.pth
Running val predictions (3926 samples)...


Running test predictions (3872 samples)...


Val: 3926 | Test: 3872


## 6. Evaluate on 200 Test Samples

Dùng `pricer_vi_2/evaluator.py` — scatter plot + error trend chart.

In [7]:
def dnn_tfidf_pricer(item):
    return runner.inference(item)

results = evaluate(dnn_tfidf_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R²: {results['r2']:.1f}%")

  0%|          | 0/200 [00:00<?, ?it/s]

17 14 27 167 22 353 90 26 69 5 13 55 395 62 138 50 12 286 151 6 27 9 7 15 46 4 119 49 520 11 53 22 2 1 6 3 41 28 139 357 16 108 135 13 1 437 8 5 7 7 43 48 193 416 73 61 81 16 55 131 109 154 39 41 96 23 1 100 105 10 4 27 10 188 7 119 122 28 49 24 106 533 162 98 19 87 51 4 91 7 4 2 52 513 11 9 18 114 9 39 231 21 46 84 112 26 56 31 18 4 46 27 204 281 94 20 23 65 65 2 328 69 3 37 34 3 39 23 100 13 75 10 106 0 30 11 273 67 29 62 45 101 47 1 34 53 143 5 6 2 323 230 9 156 12 276 3 18 8 8 11 26 98 2 19 123 41 149 53 153 160 0 26 147 23 6 71 0 131 42 37 32 7 30 9 33 10 477 352 58 135 26 19 0 2 49 19 155 13 2 


MAE: 77.3k VND | MSE: 17,223 | R²: 70.0%


## 7. Sanity Check — Load Roundtrip

In [8]:
# Inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

# Load roundtrip test
runner.load("weights/dnn_tfidf.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")

Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L 
Actual:  479.0k VND
Predict: 496.1k VND
Error:   17.1k VND

Load roundtrip PASSED. Diff: 0.0000k
